# A.1: Base and instruction-tuned models on the same requests

Load SmolLM2-135M and its instruction-tuned version, and answer three requests with greedy decoding.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)
BASE, INSTRUCT = "HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(INSTRUCT)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).eval()
inst = AutoModelForCausalLM.from_pretrained(INSTRUCT, dtype=torch.float32).eval()

def generate(model, text, max_new_tokens=40):
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=False)

def chat(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

requests = ["What is the capital of France?",
            "Give me three tips for staying focused while studying.",
            "Translate 'good morning' into Spanish."]
for r in requests:
    print("REQUEST:", r)
    print("  base:    ", repr(generate(base, r)))
    print("  instruct:", repr(generate(inst, chat([{"role": "user", "content": r}]))))


**Expected output**

Output:

```text
REQUEST: What is the capital of France?
  base:     '\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nThe capital'
  instruct: 'The capital of France is Paris.<|im_end|>'
REQUEST: Give me three tips for staying focused while studying.
  base:     '\n\nFirst, try to focus on one thing at a time. If you are studying for a test, try to focus on one question at a time. If you are studying for a presentation,'
  instruct: '1. Create a Dedicated Study Space: Choose a quiet and comfortable place where you can focus without distractions. This could be a library, a coffee shop, or even a peaceful garden.\n\n2'
REQUEST: Translate 'good morning' into Spanish.
  base:     "\n\nThe Spanish word for good morning is 'morir'.\n\nThe word for good night is 'suena'.\n\nThe word for bad morning is 'morir'.\n\nThe"
  instruct: "'Good morning' es 'Hola' in español.<|im_end|>"
```


In [ ]:
for model, name in [(base, "base"), (inst, "instruct")]:
    ids = tok(chat([{"role": "user", "content": requests[0]}]), return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
    new = out[0, ids.shape[1]:].tolist()
    stop = tok.convert_tokens_to_ids("<|im_end|>")
    print(name, "emitted <|im_end|> after", new.index(stop) + 1 if stop in new else None, "tokens")


**Expected output**

Output:

```text
base emitted <|im_end|> after None tokens
instruct emitted <|im_end|> after 8 tokens
```
